# Rule-erosion sweep

One pass: 4 models x k=0..5 x 10 seeds. Everything is resumable — if the runtime dies,
run the cells again and only missing episodes are run. Results are pushed to GitHub the
moment each model finishes, and the HTML reports are built **from** the saved results, so
they can be rebuilt any time without rerunning anything.

In [ ]:
# 1. setup (once per runtime)
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO = '/content/agent-history-effects'
if not os.path.exists(REPO):
    !git clone https://github.com/pnngocdoan/agent-history-effects {REPO}
%cd {REPO}
!git pull
sys.path.insert(0, REPO)

DRIVE = '/content/drive/MyDrive/risk_discounting/sprint/multiturn'
os.makedirs(DRIVE, exist_ok=True)

!pip -q install vllm

In [ ]:
# 2. GitHub push access (once per runtime; token is never stored)
from getpass import getpass
import subprocess

token = getpass('GitHub token (repo scope): ')
subprocess.run(['git', 'config', 'user.name', 'Ngoc Doan'], cwd=REPO)
subprocess.run(['git', 'config', 'user.email', 'pnngocdoan@users.noreply.github.com'], cwd=REPO)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/pnngocdoan/agent-history-effects.git'], cwd=REPO)
del token

def push(msg):
    r = subprocess.run(f'git add results reports && git commit -m "{msg}" && git push',
                       shell=True, cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-400:] or r.stderr[-400:])

In [ ]:
# 3. THE SWEEP — run once; rerunning only fills in what's missing
import json, time
from pathlib import Path
import serve

MODELS = [('Qwen/Qwen3-0.6B',   'qwen3-0.6b'),
          ('Qwen/Qwen3-1.7B',   'qwen3-1.7b'),
          ('Qwen/Qwen3-4B',     'qwen3-4b'),
          ('Qwen/Qwen3-8B-AWQ', 'qwen3-8b')]
ROUNDS, EPISODES = '0 1 2 3 4 5', 10
TARGET = 6 * EPISODES

def done(out):
    f = Path(out) / 'episodes.jsonl'
    return f.exists() and sum(1 for l in f.read_text().splitlines() if l.strip()) >= TARGET

for hf, name in MODELS:
    out = f'results/multiturn/{name}'
    if done(out):
        print(f'{name}: complete, skipping'); continue
    print(f'=== {name} ===', flush=True)
    !pkill -f api_server
    time.sleep(8)
    proc, log = serve.start_server(hf, name, log_dir=out)
    serve.wait_for_server(proc, log=log)
    !python multiturn/run_pilot.py --model {name} --rounds {ROUNDS} --episodes {EPISODES} \
        --workers 8 --out {out}
    proc.terminate()
    # save right away: Drive copy + GitHub push, before the next model starts
    !cp {out}/episodes.jsonl {DRIVE}/{name}.episodes.jsonl
    push(f'sweep results: {name}')
print('sweep done')

In [ ]:
# 4. HTML reports — ISOLATED: reads the saved results only. Rerun this cell as often as
#    you like (e.g. after tweaking multiturn/html_report.py) without touching the sweep.
from multiturn.html_report import render_html

for _, name in MODELS:
    out = f'results/multiturn/{name}'
    if Path(out, 'episodes.jsonl').exists():
        render_html(out, f'reports/{name}.html', model=name)
        !cp reports/{name}.html {DRIVE}/
push('html reports')

In [ ]:
# 5. tables — per model and across models (also appended to the Drive log)
from multiturn.review import review_summary, summarize_results

for _, name in MODELS:
    out = f'results/multiturn/{name}'
    if Path(out, 'episodes.jsonl').exists():
        print(f'\n===== {name} =====')
        review_summary(out, drive=DRIVE, model=name)
summarize_results('results/multiturn', drive=DRIVE)

In [ ]:
# 6. (optional) hand-run single episodes for inspection, outside the sweep
# from multiturn.review import run_and_review
# run_and_review(k=3, model='qwen3-1.7b', out='results/multiturn/qwen3-1.7b', drive=DRIVE)